(course-core-structural-operations)=

# Module 16: Structural Operations

**Welcome back, Apprentice Master.** In [Module 15: Semantic Labeling](../Common_Core/15_Semantic_Labeling.ipynb), you mastered generating human-readable nomenclature strings using `msm.get_label()`. Now we explore 3D spatial operations and coordinate manipulation using **`msm.structure`**.

Molecular systems possess 3D spatial coordinates that define their biological conformation. The **`msm.structure`** specialized module provides geometric analysis and spatial transformation functions, including spatial centering (**`msm.structure.center()`**), translation (**`msm.structure.translate()`**), spatial alignment (**`msm.structure.least_rmsd_align()`**), and Root-Mean-Square Deviation calculation (**`msm.structure.get_rmsd()`**).

(course-core-structural-operations-learning-outcomes)=
```{admonition} Learning Outcomes
:class: dropdown learning-outcomes

By the end of this module, you will be able to:
- Calculate spatial centroids and center molecular systems at the origin using `msm.structure.get_center()` and `msm.structure.center()`.
- Translate 3D spatial coordinates using unit-safe translation vectors with `msm.structure.translate()`.
- Calculate coordinate RMSD and perform rigid structural superpositions using `msm.structure.get_rmsd()` and `msm.structure.least_rmsd_align()`.
```
The optional experimental
[MolSysMT tools addon for MolSysViewer](../../user/tools/molsysviewer_tools.md)
provides fifteen read/calculation wrappers in `view.addons.molsysmt.structure`,
including centers, raw/minimum RMSD, axes, planes, minima/maxima, neighbors, contacts and RMSF.
They return native scientific results without moving either input or creating
scene objects. Native endpoint/unit repairs are received. Minimum/neighbor
corrections have authorized local source receiving before the provider commit;
coordinate transformations
still need separate receiving. See the addon guide for actual source scope.

### 1. Geometric Centering

Let's begin by importing MolSysMT, loading our T4 Lysozyme demonstration system, and converting it to a native `MolSys` object.

In [1]:
import molsysmt as msm
from molsysmt import systems

# Load T4 Lysozyme system
lysozyme = msm.convert(systems['T4 lysozyme L99A']['181l.bcif.gz'], to_form='molsysmt.MolSys')

# Calculate initial spatial center (centroid)
center_before = msm.structure.get_center(lysozyme)
print(f"Center before centering: {center_before}")

Center before centering: [[[3.494637057598893 1.1291843164469129 0.9497851492019409]]] nanometer


To center a molecular system so that its spatial centroid rests at the origin $(0, 0, 0)\text{ nm}$, use **`msm.structure.center()`**:

In [2]:
# Center system at the origin
lysozyme_centered = msm.structure.center(lysozyme)

# Verify centered coordinates
center_after = msm.structure.get_center(lysozyme_centered)
print(f"Center after centering : {center_after}")

Center after centering : [[[-3.4032457629245257e-15 -1.4200992914566887e-15   1.6802042832078705e-15]]] nanometer


:::{hint}
:class: dropdown
**msm.structure.get_center()**: Computes the geometric centroid (unweighted mean coordinates) of a molecular system or selection. See API doc: {func}`molsysmt.structure.get_center`.
:::

### 2. Spatial Translation

You can translate a molecular system or selection in 3D space by specifying a displacement vector with physical units using **`msm.structure.translate()`**:

In [3]:
# Translate system by 1.0 nm along the X axis
lysozyme_shifted = msm.structure.translate(lysozyme_centered, translation='[1.0, 0.0, 0.0] nanometers')

# Verify new center location
center_shifted = msm.structure.get_center(lysozyme_shifted)
print(f"Center after translation: {center_shifted}")

Center after translation: [[[0.9999999999999959 -1.4200992914566887e-15 1.6802042832078705e-15]]] nanometer


:::{hint}
:class: dropdown
**msm.structure.translate()**: Shifts 3D spatial coordinates of a molecular system or selection along a specified displacement vector. See API doc: {func}`molsysmt.structure.translate`.
:::

### 3. Structural Alignment and RMSD

To compare or superimpose two conformational structures, use **`msm.structure.get_rmsd()`** to measure Root-Mean-Square Deviation, and **`msm.structure.least_rmsd_align()`** to perform rigid spatial alignment:

In [4]:
# Calculate RMSD between centered and shifted systems
rmsd_val = msm.structure.get_rmsd(lysozyme_centered, reference_molecular_system=lysozyme_shifted)
print(f"RMSD before alignment: {rmsd_val}")

# Perform optimal rigid alignment (least-RMSD fit)
aligned_sys = msm.structure.least_rmsd_align(lysozyme_shifted, reference_molecular_system=lysozyme_centered)

# Verify RMSD after alignment
rmsd_after = msm.structure.get_rmsd(lysozyme_centered, reference_molecular_system=aligned_sys)
print(f"RMSD after alignment : {rmsd_after}")

RMSD before alignment: [1.0] nanometer


RMSD after alignment : [0.32478039342852333] nanometer


:::{hint}
:class: dropdown
**msm.structure.get_rmsd()**: Computes the Root-Mean-Square Deviation between two coordinate structures or structural selections. See API doc: {func}`molsysmt.structure.get_rmsd`.
:::

(course-core-structural-operations-unaligned-pose-rmsd)=
### 4. Comparing ligand poses

For receptor-relative pose evaluation you preserve the ligand's position and orientation. `msm.structure.get_symmetry_corrected_rmsd()` minimizes over chemically admitted pairings without rigid superposition. A raw positional baseline needs an explicit original map. This bundled ligand self-comparison illustrates the input contract; {ref}`Tutorial_Get_symmetry_corrected_rmsd` and {ref}`Cookbook_SymmetryPoseRMSD` cover permutations, translations, selected structures and provenance.

Partial `fixed_atom_correspondence` anchors constrain the chemical search, including unselected context atoms. Complete `candidate_atom_correspondences` restrict evaluation to validated caller-supplied maps. Keep `search_scope` with your result: a conditional minimum does not certify the unrestricted symmetry minimum. A raw baseline map does not impose either restriction.


In [5]:
import molsysmt as msm
from rdkit import Chem
molsys_reference = Chem.MolFromMolFile(msm.systems['caffeine']['caffeine.sdf'])
pose_comparison = msm.structure.get_symmetry_corrected_rmsd(molsys_reference)
assert pose_comparison['rmsd'].shape == (1,)
print('Unaligned self-reference RMSD:', pose_comparison['rmsd'])
import numpy as np
identity_map = np.column_stack((np.arange(molsys_reference.GetNumAtoms()), np.arange(molsys_reference.GetNumAtoms())))
conditional_pose = msm.structure.get_symmetry_corrected_rmsd(
    molsys_reference, selection="all", candidate_atom_correspondences=identity_map)
assert conditional_pose["search_scope"] == "supplied_atom_correspondences"
print("Declared search scope:", conditional_pose["search_scope"])


Unaligned self-reference RMSD: [0.0] nanometer


Declared search scope: supplied_atom_correspondences


:::{hint}
:class: dropdown
{func}`molsysmt.structure.get_symmetry_corrected_rmsd` returns the admitted minimum, chosen mapping index and detached scientific provenance. Use {func}`molsysmt.structure.get_least_rmsd` when optimal superposition is intended.
:::

### Preserving coordinate endpoints

Distance populations carry their own atom, weight and structure indices. Structure
lists pair positionally; explicit repeated indices reuse observations. A second
system is independent even when its atom-index labels match the first system.
Extrema return local reduction indices. Neighbors exclude physical self identity,
retain distinct coincident atoms and break distance ties by local target index.
Contacts include their cutoff boundary. Unsupported formats and incompatible
axes must reject explicitly. See {ref}`user-tools-distance-family` for the full
contract, including periodic centers and resident/disk memory limits.

:::{hint}
:class: dropdown

{func}`molsysmt.structure.get_distances`,
{func}`molsysmt.structure.get_minimum_distances`,
{func}`molsysmt.structure.get_maximum_distances`,
{func}`molsysmt.structure.get_neighbors` and
{func}`molsysmt.structure.get_contacts` preserve these endpoint conventions.
:::


In [6]:
import numpy as np
from molsysmt.native import Structures
puw = msm.pyunitwizard
xyz_endpoints = np.array([[[0., 0., 0.], [1., 0., 0.], [4., 0., 0.]],
                          [[1., 0., 0.], [2., 0., 0.], [5., 0., 0.]]])
molsys_endpoints = Structures(coordinates=puw.quantity(xyz_endpoints, 'nm'))
molsys_other = Structures(coordinates=puw.quantity(xyz_endpoints + [10., 0., 0.], 'nm'))
endpoint_distances = msm.structure.get_distances(molsys_endpoints, selection=[0],
    molecular_system_2=molsys_other, selection_2=[1, 2], pbc=False)
np.testing.assert_allclose(puw.get_value(endpoint_distances, to_unit='nm')[:, 0], [[11., 14.]] * 2)
minimum_indices, minimum_values = msm.structure.get_minimum_distances(molsys_endpoints,
    selection=[0, 1], selection_2=[1, 2], center_of_atoms=True, center_of_atoms_2=True,
    weights=[1., 1.], weights_2=[1., 3.], pbc=False)
np.testing.assert_allclose(puw.get_value(minimum_values, to_unit='nm'), [2.75, 2.75])
maximum_indices, maximum_values = msm.structure.get_maximum_distances(molsys_endpoints,
    selection=[0], selection_2=[1, 2], pbc=False)
np.testing.assert_array_equal(maximum_indices, [[0, 1], [0, 1]])
np.testing.assert_allclose(puw.get_value(maximum_values, to_unit='nm'), [4., 4.])
neighbor_indices, neighbor_values = msm.structure.get_neighbors(molsys_endpoints,
    selection=[0], molecular_system_2=molsys_other, selection_2=[0, 1], n_neighbors=1, pbc=False)
np.testing.assert_array_equal(neighbor_indices, np.zeros((2, 1, 1), dtype=int))
np.testing.assert_allclose(puw.get_value(neighbor_values, to_unit='nm'), 10.)
contact_map = msm.structure.get_contacts(molsys_endpoints, selection=[0], selection_2=[1, 2],
    threshold='1 nm', pbc=False, cell_list=False, use_gpu=False)
np.testing.assert_array_equal(contact_map, [[[True, False]], [[True, False]]])
print('Independent endpoints, weights, local indices and cutoff equality verified.')


Independent endpoints, weights, local indices and cutoff equality verified.


--- 

### 🏆 Challenge 16: The Structural Master

1. Load the **T4 Lysozyme** system (`systems['T4 lysozyme L99A']['181l.bcif.gz']`).
2. Calculate its initial spatial center using `msm.structure.get_center()`.
3. Translate the system by `[2.0, 0.0, 0.0] nm` using `msm.structure.translate()`.
4. Align the translated system back onto the original reference structure using `msm.structure.least_rmsd_align()` and verify that post-alignment RMSD is near zero.

Spatial coordinate manipulation sets the foundation for assembly and spatial positioning. In [Module 17: Merging and Growing Systems](../Common_Core/17_Merging_and_Growing_Systems.ipynb), we will learn how to combine separate molecular components and structural series.

```{key-takeaway}
The `msm.structure` module provides functions for geometric centering (`center`, `get_center`), spatial displacement (`translate`, `rotate`), and structural comparison (`get_rmsd`, `least_rmsd_align`).
```

(course-core-structural-operations-see-also)=
:::{seealso}
:class: dropdown
**API Documentation for Functions in this Module:**
- {func}`molsysmt.structure.get_center` — Spatial centroid calculation.
- {func}`molsysmt.structure.translate` — Spatial translation engine.
- {func}`molsysmt.structure.get_rmsd` — Root-Mean-Square Deviation calculator.
- {func}`molsysmt.structure.least_rmsd_align` — Least-RMSD structural alignment engine.

**Related Course Modules & Guides:**
- Previous Module: [Module 15: Semantic Labeling](../Common_Core/15_Semantic_Labeling.ipynb)
- Next Module: [Module 17: Merging and Growing Systems](../Common_Core/17_Merging_and_Growing_Systems.ipynb)
- User Guide: {ref}`user-foundations-index`
:::